In [ ]:
import numpy as np
from scipy.integrate import RK45
import matplotlib.pyplot as plt
from collections import deque
import matplotlib.animation as animation

# Constants

In [ ]:
G = 6.6743e-11 # Newton's gravitational constant

# Derivatives

$$
\begin{align*}

&\ddot{x}_{A} = \frac{G m_{B}}{r^{3}} \cdot [x_{B} - x_{A}], &&\ddot{y}_{A} = \frac{G m_{B}}{r^{3}} \cdot [y_{B} - y_{A}] \\\\

&\ddot{x}_{B} = \frac{G m_{A}}{r^{3}} \cdot [x_{A} - x_{B}], &&\ddot{y}_{B} = \frac{G m_{A}}{r^{3}} \cdot [y_{A} - y_{B}] \\\\

&r = \sqrt{(x_{B} - x_{A})^{2} + (y_{B} - y_{A})^2}

\end{align*}
$$

In [ ]:
def calculate_acceleration(m, r, pos_1, pos_2):
    return ((G * m) / r**3) * (pos_2 - pos_1)

def derivatives(t, state, mass_a, mass_b, mass_c):
    x_a, vx_a, y_a, vy_a, x_b, vx_b, y_b, vy_b, x_c, vx_c, y_c, vy_c = state



    r_ab = np.sqrt(((x_b - x_a)**2) + ((y_b - y_a)**2))
    r_ac = np.sqrt(((x_c - x_a)**2) + ((y_c - y_a)**2))
    r_bc = np.sqrt(((x_c - x_b)**2) + ((y_c - y_b)**2))

    acceleration_x_a = calculate_acceleration(mass_b, r_ab, x_a, x_b) + calculate_acceleration(mass_c, r_ac, x_a, x_c)
    acceleration_y_a = calculate_acceleration(mass_b, r_ab, y_a, y_b) + calculate_acceleration(mass_c, r_ac, y_a, y_c)

    acceleration_x_b = calculate_acceleration(mass_a, r_ab, x_b, x_a) + calculate_acceleration(mass_c, r_bc, x_b, x_c)
    acceleration_y_b = calculate_acceleration(mass_a, r_ab, y_b, y_a) + calculate_acceleration(mass_c, r_bc, y_a, y_c)

    acceleration_x_c = calculate_acceleration(mass_a, r_ac, x_c, x_a) + calculate_acceleration(mass_b, r_bc, x_c, x_b)
    acceleration_y_c = calculate_acceleration(mass_a, r_ac, y_c, y_a) + calculate_acceleration(mass_b, r_bc, y_b, y_b)

    return [
        vx_a, acceleration_x_a, vy_a, acceleration_y_a, 
        vx_b, acceleration_x_b, vy_b, acceleration_y_b,
        vx_c, acceleration_x_c, vy_c, acceleration_y_c
    ]


# Two-Body Orbit Physics

In [ ]:
mass_a = 6e+24 # approx. sun mass
mass_b = 6e+24 # approx. earth mass
mass_c = 6e+24 # approx. earth mass

initial_separation = 1.5e11 # approx. 1au (earth-sun distance)

# equilateral triangle arrangement initially
x0_a, y0_a = initial_separation / 2, 0
x0_c, y0_c = -initial_separation / 2, 0
x0_b, y0_b = 0, initial_separation * np.sqrt(3) / 2

# zero initial velocity
vx0_a, vy0_a = 0, 0
vx0_b, vy0_b = 0, 0
vx0_c, vy0_c = 0, 0

dynamical_timescale = np.sqrt(initial_separation**3 / (G * mass_a)) # approximate timescale for calculating dt
dt = dynamical_timescale / 1000 # time step

In [ ]:
solver = RK45(
    fun=lambda t, state: derivatives(t, state, mass_a, mass_b, mass_c),
    t0=0,
    y0=[
        x0_a,
        vx0_a, 
        y0_a, 
        vy0_a,

        x0_b,
        vx0_b, 
        y0_b, 
        vy0_b,

        x0_c,
        vx0_c, 
        y0_c, 
        vy0_c 
    ],
    t_bound=np.inf,
    max_step=dt
)

# Simulation

In [ ]:
%matplotlib widget

mfig, ax = plt.subplots()

# hide matplotlib widget ui
mfig.canvas.toolbar_visible = False
mfig.canvas.header_visible = False
mfig.canvas.footer_visible = False

# black background colour
ax.set_facecolor('black')
mfig.patch.set_facecolor('black')

# hide axes
ax.set_xticks([])
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)

# set x and y axes to fit orbit
x_limit = initial_separation * 1.5
y_limit = initial_separation * 1.5

ax.set_xlim(-x_limit * 1.5, x_limit * 1.5)
ax.set_ylim(-y_limit * 1.5, y_limit * 1.5)
ax.set_aspect("equal")

# plot bodies as circular markers
body_a, = ax.plot([], [], 'o', color='#ff0059', markersize=8)
body_b, = ax.plot([], [], 'o', color='#0091FF', markersize=8)
body_c, = ax.plot([], [], 'o', color='#9d00ff', markersize=8)

TRACE_LENGTH = 2000
trace_a_x = deque(maxlen=TRACE_LENGTH)
trace_a_y = deque(maxlen=TRACE_LENGTH)
trace_a, = ax.plot([], [], linewidth=0.5, color="#ff0059")

trace_b_x = deque(maxlen=TRACE_LENGTH)
trace_b_y = deque(maxlen=TRACE_LENGTH)
trace_b, = ax.plot([], [], linewidth=0.5, color="#0091FF")

trace_c_x = deque(maxlen=TRACE_LENGTH)
trace_c_y = deque(maxlen=TRACE_LENGTH)
trace_c, = ax.plot([], [], linewidth=0.5, color="#9d00ff")

def update(frame):
    global x_limit, y_limit

    for _ in range(10):
        solver.step()
        
    state = solver.y

    body_a.set_data([state[0]], [state[2]])
    body_b.set_data([state[4]], [state[6]])
    body_c.set_data([state[8]], [state[10]])

    trace_a_x.append(state[0])
    trace_a_y.append(state[2])
    trace_a.set_data(list(trace_a_x), list(trace_a_y))

    trace_b_x.append(state[4])
    trace_b_y.append(state[6])
    trace_b.set_data(list(trace_b_x), list(trace_b_y))

    trace_c_x.append(state[8])
    trace_c_y.append(state[10])
    trace_c.set_data(list(trace_c_x), list(trace_c_y))

    # adjust x and y limits if any body goes out of view
    if(np.abs(state[0]) > x_limit): 
        x_limit = np.abs(state[0])

    if(np.abs(state[2]) > y_limit): 
        y_limit = np.abs(state[2])
    
    if(np.abs(state[4]) > x_limit): 
        x_limit = np.abs(state[4])

    if(np.abs(state[6]) > y_limit): 
        y_limit = np.abs(state[6])

    if(np.abs(state[8]) > x_limit): 
        x_limit = np.abs(state[8])
    
    if(np.abs(state[10]) > y_limit): 
        y_limit = np.abs(state[10])

    ax.set_xlim(-x_limit * 1.5, x_limit * 1.5)
    ax.set_ylim(-y_limit * 1.5, y_limit * 1.5)

    return body_a, body_b, body_c, trace_a, trace_b, trace_c

anim = animation.FuncAnimation(mfig, update, interval=20, cache_frame_data=False)

plt.show()